# i+1 Story SLM — demo (base vs tuned, Colab GPU)

Runs the demo from `docs/DEMO_SCRIPT.md` as one cell per step. Each `try_model.py` call samples the
vocab, prints the selection, then generates — so the screen narrates itself. Record your screen while
running these top to bottom.

**The whole demo is one contrast:** the base model breaks the i+1 constraint; the tuned adapter holds
it. Same base, same prompt — only `--adapter` differs.


## Step 0 — GPU

**Runtime → Change runtime type → L4 GPU**, then run:

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Step 1 — clone the repo (branch `master`)

In [ ]:
REPO_URL = 'https://github.com/1624252/slm.git'
import os
if not os.path.isdir('slm'):
    !git clone --branch master $REPO_URL
%cd slm
!git pull origin master
!pip -q install -e '.[train]' bitsandbytes
print('ready')

## Step 2 — mount Drive (where the trained adapter lives)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
ADAPTER = '/content/drive/MyDrive/islm_v2_multi/qwen3_4b_v2_multi'
import os
print('adapter found:', os.path.isdir(ADAPTER))
# If False, the multilingual run didn't save here. Fall back to the en-only adapter:
#   ADAPTER = '/content/drive/MyDrive/islm_v2/qwen3_4b_v2'

## Step 3 — BASE model fails (English)

No `--adapter`. Point at the printed **TARGETS** / **KNOWN tier**, read the story, and note the
violations (out-of-vocab words, >1 new word/sentence, targets barely recur). **Run it twice** — it
breaks differently each time. That inconsistency is the point.

In [ ]:
!python scripts/try_model.py --mode en --base-path $BASE --no-think

In [ ]:
!python scripts/try_model.py --mode en --base-path $BASE --no-think

## Step 4 — TUNED model holds (English)

Same command **+ `--adapter`**. In-vocab, one new word per sentence, targets recur, still a story.
"Same base, same prompt — the only change is training data that embodies the spec."

In [ ]:
!python scripts/try_model.py --mode en --base-path $BASE --adapter $ADAPTER --no-think

## Step 5 — it generalizes: Chinese, Japanese, hard exam words

Same tuned adapter, three more modes. "Chinese, Japanese, and even GRE/SAT vocabulary — the behavior holds."

In [ ]:
!python scripts/try_model.py --mode zh --base-path $BASE --adapter $ADAPTER --no-think

In [ ]:
!python scripts/try_model.py --mode jp --base-path $BASE --adapter $ADAPTER --no-think

In [ ]:
!python scripts/try_model.py --mode en-exam --base-path $BASE --adapter $ADAPTER --no-think

## Step 6 — the numbers

Show the base-vs-tuned table to back up what the eye just saw.

In [ ]:
print(open('evals/LEADERBOARD.md', encoding='utf-8').read())